# ERA5 Data Processing

### 1. Configuration

In [ ]:
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------------- inputs ----
DATA_DIR   = Path("/home/vedant/MA605/ERA5/data")

INSTANT_GLOB = "era5_*.grib"
ACCUM_GLOB   = "era5_*.grib"

SHAPEFILE = Path("/home/vedant/MA605/India-State-and-Country-Shapefile-Updated-Jan-2020-master/India_State_Boundary.shp")
STATES    = ["Himachal Pradesh", "Uttarakhand"]

# --------------------------------------------------------------- outputs ----
OUT_NC      = Path("ERA5_daily_IST.nc")            # full-domain daily NetCDF
OUT_PARQUET = Path("ERA5_daily_IST_HP_UK.parquet") # combined-mask long table for evgam
# ------------------------------------------------------------- elevation ----

# One DEM covering the WHOLE combined HP+UK domain

DEM_PATHS   = [Path("/home/vedant/MA605/SRTM/srtm_HP_UK.tif")]   # 1 file, or several tiles
DEM_NODATA  = -32768.0          # SRTM void fill; masked before averaging
ADD_SUBGRID_RELIEF = True       # also store per-cell elevation std (orographic covariate)

# ----------------------------------------------------------------- time -----
TZ_OFFSET = pd.Timedelta(hours=5, minutes=30)          # IST
TZ_NAME   = "Asia/Kolkata (UTC+05:30)"
# ERA5 single-level accumulations are stamped at the END of the hour they cover.
ACCUM_LABEL_SHIFT = pd.Timedelta(hours=1)
EXPECTED_STEPS_PER_DAY = 24
DROP_INCOMPLETE_DAYS   = True

# ---------------------------------------------------------------- units -----
TP_TO_MM = 1000.0        # ERA5 tp is metres of water equivalent; extremes work wants mm
ALL_TOUCHED = True       # clip: keep every cell the polygon touches, not just centres

### 2. Helpers

`normalize_time` exists because cfgrib hands back different layouts depending on how the
GRIB was written: instantaneous fields usually arrive as `(time, lat, lon)` with `valid_time`
as an alias coordinate, while accumulations often arrive as a `(time, step, lat, lon)`
hypercube. Both are flattened to a single 1-D `valid_time` axis in UTC.

In [ ]:
import numpy as np
import xarray as xr

xr.set_options(keep_attrs=True)

try:
    import dask                      # noqa: F401
    CHUNKS = {}                      # lazy: one chunk per file per variable
except ImportError:
    CHUNKS = None
    print("dask not installed - everything will be loaded eagerly into RAM")

# cfgrib scalar leftovers that only clutter the output
DROP_COORDS = ("number", "surface", "heightAboveGround", "depthBelowLandLayer",
               "entireAtmosphere", "step", "time")


def normalize_time(ds):
    """Collapse whatever layout cfgrib produced into one 1-D 'valid_time' dim (UTC)."""
    if "valid_time" in ds.dims:
        out = ds

    elif "time" in ds.dims and "step" in ds.dims:
        if "valid_time" in ds.coords:
            vt = ds["valid_time"].transpose("time", "step").values
        else:
            vt = (ds["time"] + ds["step"]).transpose("time", "step").values
        out = ds.drop_vars("valid_time", errors="ignore")
        out = out.assign_coords(_vt=(("time", "step"), vt))
        out = out.stack(valid_time=("time", "step"))
        vt_flat = out["_vt"].values
        out = out.reset_index("valid_time", drop=True).drop_vars("_vt", errors="ignore")
        out = out.assign_coords(valid_time=("valid_time", vt_flat))

    elif "time" in ds.dims:
        if "valid_time" in ds.coords and ds["valid_time"].dims == ("time",):
            out = ds.swap_dims({"time": "valid_time"})
        else:
            out = ds.rename({"time": "valid_time"})

    else:
        raise ValueError(f"no recognisable time dimension in dims {dict(ds.sizes)}")

    out = out.drop_vars([c for c in DROP_COORDS if c in out.coords and c not in out.dims],
                        errors="ignore")
    return out.transpose("valid_time", ...).sortby("valid_time")


def to_local(ds, offset=TZ_OFFSET, label_shift=pd.Timedelta(0)):
    """UTC validity times -> local wall-clock; `label_shift` re-keys end-stamped accumulations."""
    return (ds.assign_coords(valid_time=ds["valid_time"] + (offset - label_shift))
              .rename({"valid_time": "time"}))


def steps_per_day(loc):
    """How many hourly timestamps fall in each local day (independent of NaN values)."""
    ones = xr.DataArray(np.ones(loc.sizes["time"], dtype="int16"),
                        coords={"time": loc["time"]}, dims="time")
    return ones.resample(time="1D").sum()


def open_stream(paths, step_type):
    """Open one cfgrib stream across all files and concatenate along hourly valid_time."""
    handles, parts = [], []
    for p in paths:
        ds = xr.open_dataset(
            p, engine="cfgrib",
            backend_kwargs={"filter_by_keys": {"stepType": step_type}, "indexpath": ""},
            decode_timedelta=True,      # silences the cfgrib FutureWarning
            chunks=CHUNKS,
        )
        handles.append(ds)
        parts.append(normalize_time(ds))

    combined = xr.concat(parts, dim="valid_time", combine_attrs="override").sortby("valid_time")

    vt = pd.DatetimeIndex(combined["valid_time"].values)
    if not vt.is_unique:                      # overlapping downloads would double-count
        dup = vt[vt.duplicated()].unique()
        raise ValueError(f"{len(dup)} duplicated hourly timestamps, e.g. {list(dup[:5])}")
    gaps = np.diff(vt.values) != np.timedelta64(1, "h")
    if gaps.any():
        print(f"  ! {gaps.sum()} gap(s) in the {step_type} hourly series, first at {vt[1:][gaps][0]}")
    return combined, handles

### 3. Load the hourly data

In [ ]:
inst_files = sorted(DATA_DIR.glob(INSTANT_GLOB))
acc_files  = sorted(DATA_DIR.glob(ACCUM_GLOB))
assert inst_files, f"no files matched {DATA_DIR / INSTANT_GLOB}"
print(f"{len(inst_files)} instant file(s), {len(acc_files)} accum file(s)")

open_handles = []
hourly_inst, h = open_stream(inst_files, "instant"); open_handles += h
hourly_acc,  h = open_stream(acc_files,  "accum");   open_handles += h

for name, ds in [("instant", hourly_inst), ("accum", hourly_acc)]:
    vt = pd.DatetimeIndex(ds["valid_time"].values)
    print(f"  {name:8s} {vt[0]} -> {vt[-1]}  ({len(vt):,} hourly steps)  vars={list(ds.data_vars)}")

lat = hourly_inst["latitude"].values
print(f"  grid: {len(lat)} lat x {len(hourly_inst['longitude'])} lon, "
      f"latitude {'descending (native ERA5)' if lat[0] > lat[-1] else 'ascending'}")

### 4. Aggregate to local calendar days

In [ ]:
# --- instantaneous fields: no label shift, they are point samples -------------
loc_inst = to_local(hourly_inst)

if {"u10", "v10"} <= set(loc_inst.data_vars):
    loc_inst["ws10"] = np.sqrt(loc_inst["u10"] ** 2 + loc_inst["v10"] ** 2)

if {"t2m", "d2m"} <= set(loc_inst.data_vars):
    # August-Roche-Magnus, saturation over water (Alduchov & Eskridge 1996):
    #   RH = 100 * e_s(Td) / e_s(T),  e_s(Tc) = C*exp(a*Tc/(b+Tc)),  Tc in degC
    a, b = 17.625, 243.04
    t_c  = loc_inst["t2m"] - 273.15
    td_c = loc_inst["d2m"] - 273.15
    rh = 100.0 * np.exp(a * td_c / (b + td_c) - a * t_c / (b + t_c))
    loc_inst["rh"] = rh.clip(0.0, 100.0)   # trim tiny numerical overshoots past 100%

# plain-named vars are daily MEANS (like sp/u10/v10); ws10 and rh join that group
mean_vars = [v for v in ("sp", "u10", "v10", "ws10", "rh") if v in loc_inst]
mm_vars   = [v for v in ("t2m", "d2m") if v in loc_inst]

parts = []
if mean_vars:
    parts.append(loc_inst[mean_vars].resample(time="1D").mean())
for v in mm_vars:
    r = loc_inst[[v]].resample(time="1D")
    parts.append(r.mean().rename({v: f"{v}_mean"}))
    parts.append(r.max().rename({v: f"{v}_max"}))
    parts.append(r.min().rename({v: f"{v}_min"}))

daily_inst = xr.merge(parts)
daily_inst["n_steps_instant"] = steps_per_day(loc_inst)

# --- accumulated field: re-key to the START of each accumulation hour ---------
loc_acc = to_local(hourly_acc, label_shift=ACCUM_LABEL_SHIFT)
daily_acc = loc_acc[["tp"]].resample(time="1D").sum(min_count=1)   # min_count: all-NaN -> NaN
daily_acc["n_steps_accum"] = steps_per_day(loc_acc)

ds_all = xr.merge([daily_inst, daily_acc])
print(ds_all)

### 5. Completeness check

In [ ]:
counts = pd.DataFrame(
    {k: ds_all[k].values for k in ("n_steps_instant", "n_steps_accum") if k in ds_all},
    index=pd.DatetimeIndex(ds_all["time"].values, name="time"),
)
incomplete = counts[(counts != EXPECTED_STEPS_PER_DAY).any(axis=1)]

print(f"{len(counts):,} local days, {len(incomplete)} incomplete:")
print(incomplete.to_string() if len(incomplete) else "  (none)")

edges = {counts.index[0], counts.index[-1]}
interior = [d for d in incomplete.index if d not in edges]
if interior:
    print(f"\n  ! {len(interior)} INTERIOR incomplete day(s) - these indicate missing hours "
          f"in the source files, not an edge effect:")
    print("   ", [str(d.date()) for d in interior[:10]], "..." if len(interior) > 10 else "")

if DROP_INCOMPLETE_DAYS and len(incomplete):
    ds_all = ds_all.sel(time=counts.index[(counts == EXPECTED_STEPS_PER_DAY).all(axis=1)])
    t = pd.DatetimeIndex(ds_all["time"].values)
    print(f"\ndropped {len(incomplete)} day(s) -> {t[0].date()} .. {t[-1].date()} ({len(t):,} days)")

### 6. Metadata, units, and write

In [ ]:
ds_all["tp"] = ds_all["tp"] * TP_TO_MM

UNITS = {"tp": ("mm", "daily total precipitation"),
         "sp": ("Pa", "daily mean surface pressure"),
         "u10": ("m s-1", "daily mean 10 m eastward wind"),
         "v10": ("m s-1", "daily mean 10 m northward wind")}
for stat, word in [("mean", "mean"), ("max", "maximum"), ("min", "minimum")]:
    UNITS[f"t2m_{stat}"] = ("K", f"daily {word} 2 m temperature")
    UNITS[f"d2m_{stat}"] = ("K", f"daily {word} 2 m dewpoint temperature")
UNITS["ws10"] = ("m s-1", "daily mean 10 m wind speed")
UNITS["rh"]   = ("%", "daily mean 2 m relative humidity (over water, Magnus)")

for v, (u, long_name) in UNITS.items():
    if v in ds_all:
        ds_all[v].attrs.update(units=u, long_name=long_name)

ds_all.attrs.update({
    "title": "ERA5 hourly aggregated to local (IST) calendar days",
    "source": "ECMWF ERA5 reanalysis, single levels, hourly",
    "timezone": TZ_NAME,
    "time_convention": ("timestamps are LOCAL IST wall-clock, not UTC; "
                        "they are stored as naive datetimes"),
    "accumulation_window": ("tp for day D is the sum of the ERA5 hourly accumulations whose "
                            "hour BEGINS within D, i.e. 00:30 IST on D to 00:30 IST on D+1"),
    "instantaneous_sampling": "24 hourly samples per day at 00:30, 01:30, ... 23:30 IST",
    "history": f"created {pd.Timestamp.now():%Y-%m-%d} from {len(inst_files)} GRIB file(s)",
})

# n_steps_* have no spatial dims, so make them coordinates: rio.clip only iterates
# data variables and would otherwise raise on them.
ds_all = ds_all.set_coords([c for c in ("n_steps_instant", "n_steps_accum") if c in ds_all])

enc = {v: {"zlib": True, "complevel": 4, "dtype": "float32"} for v in ds_all.data_vars}
enc["time"] = {"units": "days since 1900-01-01", "calendar": "proleptic_gregorian",
               "dtype": "int32"}

try:
    ds_all.to_netcdf(OUT_NC, format="NETCDF4", encoding=enc)
    print(f"wrote {OUT_NC}  ({OUT_NC.stat().st_size / 1e6:.1f} MB)")
finally:
    for h in open_handles:
        h.close()
    print(f"closed {len(open_handles)} GRIB handle(s)")

ds_all = xr.open_dataset(OUT_NC)     # reopen the written file for the rest of the notebook

### 7. Sanity checks

In [ ]:
t = pd.DatetimeIndex(ds_all["time"].values)
assert t.is_unique, "duplicate daily timestamps"
assert t.is_monotonic_increasing, "time axis not sorted"
# assert (np.diff(t.values).astype("timedelta64[D]").astype(int) == 1).all(), "non-daily spacing"
print(f"time axis ok: {t[0].date()} .. {t[-1].date()}, {len(t):,} contiguous days\n")

per_year = pd.Series(1, index=t).groupby(t.year).sum()
print("days per year (365/366 expected except at the trimmed ends):")
print(per_year.to_string(), "\n")

area_mean = ds_all["tp"].mean(dim=("latitude", "longitude"))
annual = area_mean.groupby("time.year").sum().to_pandas()
print("domain-mean annual precipitation (mm):")
print(annual.round(0).to_string())
print(f"\nmedian annual total: {annual.median():.0f} mm")
if annual.median() > 4000:
    print("  ! implausibly high - is this ERA5-Land? see the note above")
print(f"wettest single grid-day: {float(ds_all['tp'].max()):.1f} mm")
print(f"t2m_max range: {float(ds_all['t2m_max'].min()):.1f} .. {float(ds_all['t2m_max'].max()):.1f} K")
if "ws10" in ds_all:
    print(f"ws10 (daily mean wind speed): {float(ds_all['ws10'].min()):.2f} .. "
          f"{float(ds_all['ws10'].max()):.2f} m/s")
if "rh" in ds_all:
    print(f"rh (daily mean RH): {float(ds_all['rh'].min()):.1f} .. "
          f"{float(ds_all['rh'].max()):.1f} %")
assert (ds_all["t2m_max"] >= ds_all["t2m_min"]).all(), "t2m_max < t2m_min somewhere"

## 7b. Add elevation (area-averaged onto the ERA5 grid)

In [ ]:
import rioxarray as rxr
from rioxarray.merge import merge_arrays
from rasterio.enums import Resampling

# --- load DEM(s), masking void fill so it can't contaminate the average -------
dems = []
for p in DEM_PATHS:
    da = rxr.open_rasterio(p, masked=True).squeeze(drop=True)   # masked=True -> nodata to NaN
    if da.rio.nodata is None:
        da = da.rio.write_nodata(DEM_NODATA)
    da = da.where(da > DEM_NODATA + 1)                          
    dems.append(da)

dem = dems[0] if len(dems) == 1 else merge_arrays(dems)        # merge tiles BEFORE regridding
if dem.rio.crs is None:
    dem = dem.rio.write_crs("EPSG:4326")
elif dem.rio.crs.to_epsg() != 4326:
    dem = dem.rio.reproject("EPSG:4326")
print(f"DEM: {dem.rio.crs}, {dem.shape[-2]}x{dem.shape[-1]} px, "
      f"range {float(dem.min()):.0f}-{float(dem.max()):.0f} m")

# target grid = the full ERA5 domain (elevation gets clipped alongside the rest below)
ds_all = ds_all.rio.write_crs("EPSG:4326").rio.set_spatial_dims("longitude", "latitude")

def regrid(src, how):
    out = src.rio.reproject_match(ds_all, resampling=how)
    if "x" in out.dims:
        out = out.rename({"x": "longitude", "y": "latitude"})
    return out.assign_coords(longitude=ds_all["longitude"], latitude=ds_all["latitude"])

ds_all["elevation"] = regrid(dem, Resampling.average)         # mean elevation per cell
ds_all["elevation"].attrs.update(units="m", long_name="cell-mean surface elevation (SRTM)")

if ADD_SUBGRID_RELIEF:
    # std within each cell: sqrt(E[x^2] - E[x]^2) via two averagings
    mean_sq = regrid(dem ** 2, Resampling.average)
    var = (mean_sq - ds_all["elevation"] ** 2).clip(min=0)
    ds_all["elevation_std"] = np.sqrt(var)
    ds_all["elevation_std"].attrs.update(units="m",
        long_name="within-cell elevation std (sub-grid relief, SRTM)")

# report coverage; cells whose entire footprint was void come back NaN
n_missing = int(ds_all["elevation"].isnull().sum())
print(f"elevation attached to the grid; {n_missing} cell(s) with no DEM coverage"
      + (" (fill from neighbours or ERA5 orography before modelling)" if n_missing else ""))
del dem, dems

### 8. Clip to the combined region and export for evgam

The two states are spatially contiguous and will be modelled with shared smooths, so
they are dissolved into a **single mask** and clipped **once**.

The result is written as a **long-format Parquet table** (one row per in-region cell-day),
which is the shape `evgam` expects. `longitude`/`latitude` stay as numeric columns for an
`s(lon, lat)` spatial smooth, and `time` is written as a plain `Date`.

In [ ]:
import geopandas as gpd

gdf = gpd.read_file(SHAPEFILE)
gdf = gdf[gdf.State_Name.isin(STATES)].copy()          # .copy() -> real frame, not a view
assert len(gdf) == len(STATES), f"found {list(gdf.State_Name)}, expected {STATES}"

if gdf.crs is None:
    raise ValueError("shapefile has no CRS; set it explicitly before continuing")
if gdf.crs.to_epsg() != 4326:
    print(f"reprojecting from {gdf.crs.to_string()} to EPSG:4326")
    gdf = gdf.to_crs("EPSG:4326")

invalid = ~gdf.geometry.is_valid
if invalid.any():
    print(f"repairing {int(invalid.sum())} invalid geometry(ies): {list(gdf.loc[invalid, 'State_Name'])}")
    gdf.loc[:, "geometry"] = gdf.geometry.make_valid()   # .loc, not chained assignment
    assert gdf.geometry.is_valid.all(), "geometry repair failed"
else:
    print("all geometries valid")

print(gdf[["State_Name"]].assign(bounds=gdf.geometry.bounds.round(2).values.tolist()).to_string(index=False))

region = gdf.dissolve()
region["State_Name"] = " + ".join(sorted(gdf["State_Name"]))
print(f"dissolved {len(gdf)} states -> 1 region: {region['State_Name'].iloc[0]}")


In [ ]:
import rioxarray as rxr   # noqa: F401  (registers the .rio accessor)
import pyarrow as pa, pyarrow.parquet as pq

ds_all = ds_all.rio.write_crs("EPSG:4326")
ds_all = ds_all.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
xmin, ymin, xmax, ymax = ds_all.rio.bounds()
print(f"dataset bounds: {xmin:.3f}, {ymin:.3f}, {xmax:.3f}, {ymax:.3f}")

# --- clip once to the combined region -----------------------------------------
clipped = ds_all.rio.clip(region.geometry, crs="EPSG:4326",
                          drop=True, all_touched=ALL_TOUCHED)
n_cells = int(clipped["tp"].isel(time=0).notnull().sum())
assert n_cells > 0, "clip returned no valid cells - check CRS / bounds overlap"
print(f"kept {n_cells} grid cells across the combined region "
      f"({clipped.sizes['latitude']} lat x {clipped.sizes['longitude']} lon before masking)")

# --- melt to long format: one row per in-region cell-day ----------------------
df = clipped.to_dataframe().reset_index()
df = df.dropna(subset=["tp"]).reset_index(drop=True)

df["time"] = pd.to_datetime(df["time"]).dt.date

drop = ("spatial_ref", "n_steps_instant", "n_steps_accum")
value_cols = [c for c in df.columns
              if c not in ("time", "latitude", "longitude", *drop)]
df = df[["time", "latitude", "longitude", *value_cols]]

# df.to_parquet(OUT_PARQUET, index=False)
pq.write_table(pa.Table.from_pandas(df, preserve_index=False),
               OUT_PARQUET, compression="snappy")
ds_all.close()
print(f"wrote {OUT_PARQUET}  ({OUT_PARQUET.stat().st_size / 1e6:.1f} MB)")
print(f"  {len(df):,} rows  |  {df['time'].nunique():,} days x {n_cells} cells")
print(f"  columns: {list(df.columns)}")
print(df.head(3).to_string(index=False))